# Inference on New Vestibular Schwannoma Cases

This notebook segments new vestibular schwannoma (VS) MRI scans using the models trained in `01_five_fold_cross_validation.ipynb` and saves the predicted tumor masks as NIfTI files. The audience is students and researchers, so every step is explained.

It runs a **5-fold soft-voting ensemble**: all five cross-validation models for the chosen architecture are run on the scan, their class probabilities are averaged, and the averaged map is decoded into a mask. This is the standard way to deploy a cross-validation and usually beats any single fold.

## What this notebook does

1. Locates the five fold checkpoints for a model (UNet, DynUNet, or SegMamba) trained in notebook 01.
2. Rebuilds the exact preprocessing used at training time.
3. Runs sliding-window patch inference with each fold and averages the probabilities (soft voting).
4. Saves and visualizes the predicted mask, and optionally scores it against ground truth.

## Sliding-window patch inference

VS scans are large 3D volumes, too big to feed to a 3D network in one piece. Instead we use **patch-based inference**:

- A `GridSampler` slides a fixed-size window (the patch) across the volume with overlap, producing many small patches.
- The network predicts on each patch.
- A `GridAggregator` stitches the patch predictions back into a full-volume prediction. We use **Hann-window** aggregation, which weights each patch by a smooth cosine taper so overlapping patches blend without visible seams at the patch borders.

fastMONAI wraps this in `PatchInferenceEngine` and the `patch_inference` helper.

## Soft-voting ensemble

A 5-fold cross-validation produces five models, each trained on a different 4/5 of the data. To turn them into one predictor we combine them by **soft voting**: run all five, apply softmax to each output, average the five probability maps, and only then take the argmax. Averaging probabilities (not the decoded masks) is the correct way to combine multi-class segmentation models, and it gives a small, reliable accuracy gain over any single fold. The engine returns each fold's probability map already resampled and reoriented into the input's original space, so the five maps are voxel-aligned and can be averaged directly.

## The one rule that matters most: preprocessing parity

A segmentation model produces correct output only when the data it sees at inference is preprocessed **identically** to the data it saw during training. Three settings must match exactly:

- **`apply_reorder`**: reorientation to RAS+ canonical orientation.
- **`target_spacing`**: the voxel spacing the volume is resampled to.
- **`normalization`**: the intensity normalization (here, foreground Z-normalization).

A mismatch does not raise an error. It silently produces wrong predictions, for example masks that are shifted, mirrored, or rotated. This notebook reuses the same `PatchConfig` and normalization as training so the two pipelines stay in lockstep.

## 1. Environment setup

We import the fastMONAI public API and pin the working directory to the project folder so that the relative dataset paths used here (for example `../nii_data/...`) resolve the same way they do during training.

`from fastMONAI.vision_all import *` brings in everything we need for inference: `PatchConfig`, `patch_inference`, `ZNormalization`, `store_patch_variables` / `load_patch_variables`, `MedImage`, `MedMask`, and the metric functions. `load_learner` comes from fastai and is used to load an exported model.

In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
import torchio as tio

from fastMONAI.vision_all import *
from fastai.learner import load_learner

# Resolve data paths against the repo root so they work wherever the kernel started
# (VS Code launches at the workspace root; plain Jupyter, in the notebook folder).
import fastMONAI
REPO_ROOT = Path(fastMONAI.__file__).resolve().parent.parent
os.chdir(REPO_ROOT / "research" / "vestibular_schwannoma")

import fastai, monai  # fastMONAI already imported above
print("fastMONAI:", fastMONAI.__version__)
print("fastai:   ", fastai.__version__)
print("MONAI:    ", monai.__version__)
print("TorchIO:  ", tio.__version__)
print("PyTorch:  ", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Configuration

Everything you would normally change lives in the cell below. Treat this notebook as a **template**: pick the architecture with `MODEL_KEY`, list the scans in `NEW_CASES`, and run top to bottom. The notebook always soft-votes the five cross-validation folds for the chosen model.

| Knob | Meaning |
| --- | --- |
| `MODEL_KEY` | Which architecture to load: `"unet"`, `"dynunet"`, or `"segmamba"`. Must match a trained CV experiment. |
| `FOLD_LEARNER_PATHS` | Optional `{fold: path/to/best_learner.pkl}`. Leave empty to auto-discover the five folds from MLflow. |
| `NEW_CASES` | List of NIfTI image paths to segment. These are **raw** scans; the pipeline preprocesses them for you. |
| `OUTPUT_DIR` | Where predicted masks are written as NIfTI. |
| `USE_TTA` | 8-flip test-time augmentation. More robust, roughly 8x slower. |
| `USE_AMP` | Automatic mixed precision (float16) forward pass. Faster on CUDA, ignored on CPU. |

`TARGET_SPACING` and `PATCH_SIZE` are part of the shared contract with training and must not be changed independently of notebook 01.

In [ ]:
# --- Which model ---------------------------------------------------------------
MODEL_KEY = "unet"          # "unet" | "dynunet" | "segmamba"  (must match a trained CV run)

# Ensemble checkpoints: {fold_number: path/to/best_learner.pkl}. Leave empty to
# auto-discover the five folds from the MLflow experiment "vs5f_<MODEL_KEY>" below.
FOLD_LEARNER_PATHS = {}

# --- What to segment and where to write it -------------------------------------
NEW_CASES = [
    "../nii_data/queen_square_data/vs_gk_1/vs_gk_1_t1_refT1.nii.gz",
]
OUTPUT_DIR = "inference_predictions"

# --- Inference options ---------------------------------------------------------
USE_TTA = True     # 8-flip test-time augmentation (more robust, ~8x slower)
USE_AMP = True     # mixed precision on CUDA (ignored on CPU)

# --- Shared contract with training notebook 01 (do not change in isolation) ----
TARGET_SPACING = [0.4102, 0.4102, 1.5]
PATCH_SIZE = [192, 192, 48]

print(f"Model: {MODEL_KEY} | mode: 5-fold soft-voting ensemble")
print(f"Cases to segment: {len(NEW_CASES)}")

### Locate the fold checkpoints

Notebook 01 logs `best_learner.pkl` as an MLflow artifact for every fold, under experiments named `vs5f_unet`, `vs5f_dynunet`, and `vs5f_segmamba` (one run per fold, tagged with its fold number). The helper below finds one checkpoint per fold for `MODEL_KEY` and returns `{fold: local_path}`. It is best-effort and never raises, so it is safe to run before any training exists.

If you already have the paths, set `FOLD_LEARNER_PATHS` in the configuration cell and this lookup is skipped.

In [ ]:
# find_fold_learners now lives in the library (fastMONAI.utils) and is imported via
# `from fastMONAI.vision_all import *` above; the experiment name is passed explicitly
# (it also respects a user-configured MLflow tracking server, unlike the old inline copy).

if not FOLD_LEARNER_PATHS:
    FOLD_LEARNER_PATHS = find_fold_learners(f"vs5f_{MODEL_KEY}")

print("Fold checkpoints:", {k: str(v) for k, v in sorted(FOLD_LEARNER_PATHS.items())} or "(none)")

## 3. Build the inference configuration

`PatchConfig` is the single object that describes how patches are sampled, how they are aggregated, and (critically) how the raw input is preprocessed. We build the **exact same** config that training used. The values below are the shared contract with notebook 01.

A few points worth understanding:

- **`preprocessed=True`** only affects *training* (it tells the training pipeline that the data on disk was already reordered, resampled, and normalized, so it should not do it again). At *inference* the engine always reorders, resamples, and normalizes the raw input for you, so we point it at the original scans.
- **`patch_overlap=0.5`** means neighboring windows overlap by half a patch. More overlap gives smoother, more accurate borders at the cost of speed.
- **`aggregation_mode="hann"`** blends overlapping patches with a Hann taper (smoothest transitions).
- **`keep_largest_component=True`** keeps only the largest connected foreground blob, a simple and effective post-processing step for a single-tumor task like VS.

The pre-inference normalization is defined separately as `pre_inference_tfms` and passed to `patch_inference`. It must be the same transform training used: foreground `ZNormalization`. Passing it explicitly makes the parity obvious and overrides whatever is stored on the config.

In [ ]:
# Pre-inference intensity normalization. MUST match training exactly.
pre_inference_tfms = [ZNormalization(masking_method="foreground")]

# The same PatchConfig used for training in notebook 01.
patch_config = PatchConfig(
    patch_size=PATCH_SIZE,
    samples_per_volume=4,
    sampler_type="label",
    label_probabilities={0: 0.2, 1: 0.8},
    patch_overlap=0.5,
    keep_largest_component=True,
    target_spacing=TARGET_SPACING,
    preprocessed=True,          # only affects training; inference always preprocesses raw input
    aggregation_mode="hann",
    queue_num_workers=16,
    queue_length=1200,
)
print(patch_config)

### Persisting and reloading the config

To guarantee parity across machines and over time, the patch config can be written to a small JSON file with `store_patch_variables(...)` and read back with `load_patch_variables(...)`. The JSON captures `patch_size`, `target_spacing`, `apply_reorder`, the sampler settings, and the `normalization` spec, so an inference run can reconstruct the training config without re-typing it. This is the recommended way to ship a config alongside a checkpoint.

The cell below writes the current config out and reads it back to confirm the round-trip. An inference run on another machine would simply start from `load_patch_variables(...)`.

In [ ]:
CONFIG_JSON = "inference_patch_config.json"

# Coerce the normalization transforms to a JSON-serializable spec. PatchConfig does this
# coercion in __post_init__, so we build a throwaway config to obtain the spec list.
norm_specs = PatchConfig(patch_size=PATCH_SIZE, normalization=pre_inference_tfms).normalization

# Persist the config so any inference run can reconstruct it verbatim.
store_patch_variables(
    CONFIG_JSON,
    patch_size=patch_config.patch_size,
    patch_overlap=patch_config.patch_overlap,
    aggregation_mode=patch_config.aggregation_mode,
    apply_reorder=patch_config.apply_reorder,
    target_spacing=patch_config.target_spacing,
    sampler_type=patch_config.sampler_type,
    label_probabilities=patch_config.label_probabilities,
    samples_per_volume=patch_config.samples_per_volume,
    queue_length=patch_config.queue_length,
    queue_num_workers=patch_config.queue_num_workers,
    keep_largest_component=patch_config.keep_largest_component,
    normalization=norm_specs,
)

# Read it back to confirm the round-trip (inference elsewhere would start here).
reloaded = load_patch_variables(CONFIG_JSON)
print("Persisted config to", CONFIG_JSON)
print("target_spacing:", reloaded["target_spacing"])
print("patch_size:    ", reloaded["patch_size"])
print("normalization: ", reloaded["normalization"])

## 4. Load the models

We load the five fold learners for `MODEL_KEY` into a list. Each exported learner carries its own trained weights and preprocessing, so no architecture code is needed. `load_learner` uses Python `pickle`, which can execute arbitrary code, so only load checkpoints you trust.

The list of learners is what the inference engine soft-votes in the next section.

In [ ]:
def _load_learner(pkl_path):
    """Load an exported fastai learner onto GPU if available, else CPU; set eval mode."""
    learn = load_learner(pkl_path, cpu=not torch.cuda.is_available())
    learn.model.eval()
    return learn


# Load the five fold learners for MODEL_KEY into a list. Each exported learner carries its
# own trained weights and preprocessing, so no architecture code is needed here. load_learner
# uses Python pickle, which can execute arbitrary code, so only load checkpoints you trust.
assert FOLD_LEARNER_PATHS, (
    "No fold checkpoints found. Train the folds with notebook 01, or set "
    "FOLD_LEARNER_PATHS = {1: '.../best_learner.pkl', ...} in the config cell.")

predictors = []
for fold in sorted(FOLD_LEARNER_PATHS):
    predictors.append(_load_learner(FOLD_LEARNER_PATHS[fold]))
    print(f"Loaded fold {fold}: {FOLD_LEARNER_PATHS[fold]}")
print(f"Ensemble ready: {len(predictors)} fold model(s) for '{MODEL_KEY}'.")

### A note on SegMamba

SegMamba is not part of MONAI; it comes from our fork **skaliy/SegMamba-V2**. Install it into this environment before selecting `MODEL_KEY = "segmamba"`:

```bash
pip install -e /home/sathiesh/ml_projects/SegMamba-V2          # base (import path: models_segmamba)
pip install -e "/home/sathiesh/ml_projects/SegMamba-V2[gpu]"   # adds mamba-ssm + causal-conv1d for CUDA
```

The model is imported as `from models_segmamba.segmambav2 import SegMamba`. It supports two Mamba backends:

- **`mamba_backend="mamba_ssm"`** (default): the CUDA kernel used for training and fast GPU inference.
- **`mamba_backend="mambamixer"`**: a pure-PyTorch backend (needs `transformers`) that runs on CPU. It is weight-compatible with a `mamba_ssm`-trained checkpoint, so the same `.pth` loads with `strict=True` and no retraining.

UNet and DynUNet need no fork and run on CPU or GPU as-is. One environment caveat: `causal-conv1d` and `mamba-ssm` must be compiled against the active PyTorch build, otherwise the import fails with an "undefined symbol" ABI error. On CPU we sidestep this with a small `_force_cpu_mamba_backend()` helper (defined in the section 8 cell below, mirroring `infer_segmamba_cpu.py`) that tells `transformers` not to probe the CUDA kernels. Section 8 shows the CPU path end to end.

## 5. Run inference

Ensembling is built into fastMONAI: pass a **list of learners** to `patch_inference` and it
soft-votes them, averaging each patch's class probabilities across the models before taking the
argmax, all in a single sliding-window pass. We pass the list of fold learners as
`learner = predictors`.

For each scan `patch_inference` reorders, resamples, and normalizes the raw input exactly as in
training, slides the patch grid, aggregates with the Hann window (soft-voting per patch across the
folds), resamples the prediction back to the input's original grid, and applies the training-time
post-processing (argmax and keep-largest-component). Predictions are written to `OUTPUT_DIR` as
`<name>_pred.nii.gz` and returned as in-memory masks in `predictions` for the sections below.

Two options:

- **`tta=USE_TTA`**: each patch is run through 8 axis-flip combinations and the probabilities
  averaged (combined with the fold averaging). Tighter borders, roughly 8x compute. Impractical on CPU.
- **`amp=USE_AMP`**: float16 forward pass on CUDA. Faster and lighter; ignored on CPU.

To obtain the averaged probability map instead of a mask, pass `return_probabilities=True`.

In [ ]:
# Ensembling is a fastMONAI feature: a list of learners makes patch_inference soft-vote them
# (per-patch probabilities averaged before argmax) in one sliding-window pass. Post-processing
# (argmax, keep-largest-component) is applied to the averaged map.
learners = predictors

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
predictions = patch_inference(
    learner=learners,
    config=patch_config,
    file_paths=NEW_CASES,
    pre_inference_tfms=pre_inference_tfms,
    save_dir=OUTPUT_DIR,
    progress=True,
    tta=USE_TTA,
    amp=USE_AMP,
)


def _pred_filename(input_path):
    """Name patch_inference writes for a prediction: '<stem>_pred.nii[.gz]'. Mirrors the
    engine's saver, stripping only the trailing extension so '.nii' and '.nii.gz' inputs
    both map to the right output name (used again when we reload the mask to visualize)."""
    p = Path(input_path)
    if p.suffix == ".gz" and p.stem.endswith(".nii"):
        return f"{p.stem[:-4]}_pred.nii.gz"
    if p.suffix == ".nii":
        return f"{p.stem}_pred.nii"
    return f"{p.stem}_pred.nii.gz"


print(f"\nWrote {len(predictions)} prediction(s) to {OUTPUT_DIR}/ "
      f"(soft-vote ensemble of {len(predictors)} folds).")
for p in NEW_CASES:
    print(f"  {p}  ->  {OUTPUT_DIR}/{_pred_filename(p)}")

## 6. Visualize a prediction

Now we inspect a result qualitatively. `vision_plot` is imported separately (it is not part of `vision_all`). We load the input scan as a `MedImage` and its predicted mask as a `MedMask`.

Both are read with fastMONAI's default loader, which does not reorder or resample, so they share the input's original voxel grid and line up slice for slice. We pick the axial slice where the predicted tumor is largest with `find_max_slice`, then show the input, the mask, and an overlay. `voxel_size=TARGET_SPACING` sets only the display aspect ratio; it does not alter the data.

In [ ]:
from fastMONAI.vision_plot import *   # fastMONAI plotting; imported separately from vision_all
from torchio.visualization import rotate
import matplotlib.pyplot as plt

idx = 0
img_fn = NEW_CASES[idx]
# Same name patch_inference used when saving (see _pred_filename in the run-inference cell),
# so the mask reloads correctly for both .nii and .nii.gz inputs.
pred_fn = Path(OUTPUT_DIR) / _pred_filename(img_fn)

# Default fastMONAI loader: no reorder, no resample, so input and mask share the
# input's original voxel grid and align slice for slice.
img = MedImage.create(img_fn)
pred_mask = MedMask.create(pred_fn)

plane = 2  # 0 = sagittal, 1 = coronal, 2 = axial
sl = int(find_max_slice(pred_mask.data[0].cpu().numpy(), plane))
print(f"Predicted foreground voxels: {int(pred_mask.data.sum())}")
print(f"Showing plane={plane} (axial), slice={sl}")


def _disp_slice(vol3d, i, plane, vsize):
    """Replicate fastMONAI's show slicing so the overlay lines up with .show()."""
    sr, sa, ss = vsize
    ops = {0: (vol3d[i, :, :], ss / sa),
           1: (vol3d[:, i, :], ss / sr),
           2: (vol3d[:, :, i], sa / sr)}
    sl2d, aspect = ops[plane]
    return rotate(sl2d, radiological=True, n=1), aspect


fig, axes = plt.subplots(1, 3, figsize=(15, 5))
img.show(ctx=axes[0], anatomical_plane=plane, slice_index=sl, voxel_size=TARGET_SPACING)
axes[0].set_title("Input T1")
pred_mask.show(ctx=axes[1], anatomical_plane=plane, slice_index=sl, voxel_size=TARGET_SPACING)
axes[1].set_title("Predicted mask")

img_slice, aspect = _disp_slice(img.data[0].cpu().numpy(), sl, plane, TARGET_SPACING)
msk_slice, _ = _disp_slice(pred_mask.data[0].cpu().numpy(), sl, plane, TARGET_SPACING)
axes[2].imshow(img_slice, cmap="gray", aspect=aspect)
axes[2].imshow(np.ma.masked_where(msk_slice == 0, msk_slice),
               cmap="autumn", alpha=0.5, aspect=aspect)
axes[2].set_title("Overlay")
axes[2].axis("off")
plt.tight_layout()
plt.show()

## 7. Optional: validate against ground truth

If a case has an expert segmentation, we can score the prediction with the **same metric functions** notebook 01 uses in its cross-validation, so the numbers here are directly comparable. This cell is optional: it only runs when you set `GT_PATH` to a mask file.

The metrics:

- **DSC** (Dice) and **sensitivity** / **precision**: overlap-based agreement.
- **LDR** (lesion detection rate) and **signed RVE** (relative volume error).
- **Surface metrics in millimeters** via `calculate_surface_metrics`: ASSD, HD95, and NSD. These are spacing-aware, so we read the per-case voxel spacing straight from the ground-truth file with `tio.LabelMap(GT_PATH).spacing`. This matters because the VS cohort has non-uniform spacing, and using one global spacing would bias the surface distances.

The prediction and ground truth are compared as 5D tensors `[batch, channel, X, Y, Z]`, matching the metric API.

In [ ]:
# Set to a ground-truth mask to score the first case, e.g.
# GT_PATH = "../nii_data/queen_square_data/vs_gk_1/vs_gk_1_seg_refT1.nii.gz"
GT_PATH = None

if GT_PATH:
    # Score the first prediction with the library's per-case panel (loads the GT mask's
    # data and spacing from one object).
    row = evaluate_segmentations([predictions[0]], [GT_PATH]).iloc[0]
    print(f"DSC:          {row['dsc']:.4f}")
    print(f"Sensitivity:  {row['sensitivity']:.4f}")
    print(f"Precision:    {row['precision']:.4f}")
    print(f"LDR:          {row['ldr']:.4f}")
    print(f"Signed RVE:   {row['rve']:.4f}")
    print(f"ASSD (mm):    {row['assd_mm']}")
    print(f"HD95 (mm):    {row['hd95_mm']}")
    print(f"NSD tau=1mm:  {row['nsd_tau1.0_mm']}")
    print(f"Spacing (mm): {row['spacing_mm']}  | status: {row['surface_status']}")
else:
    print("GT_PATH is None; skipping ground-truth evaluation.")

## 8. Running SegMamba on CPU

When no CUDA GPU is available, SegMamba can still run through the exact same `patch_inference` pipeline by switching to the `mambamixer` backend. This mirrors `research/vs_seg/infer_segmamba_cpu.py` and `research/vs_seg/SEGMAMBA_CPU.md`.

The key points:

- Construct with `mamba_backend="mambamixer"` (pure PyTorch, needs `transformers`). No CUDA and no `mamba_ssm` required.
- A checkpoint trained with the default `mamba_ssm` backend loads into the `mambamixer` model unchanged (`strict=True`), so no retraining is needed. Correctness was verified decision-identical to GPU on a real case (Dice 1.0).
- It is slow: roughly 11 minutes per volume on CPU, and TTA is impractical there, so keep `tta=False`.

The cell below is a self-contained CPU inference path. It is guarded so it does nothing unless `RUN_SEGMAMBA_CPU` is set to `True` and a checkpoint is present.

In [ ]:
def _force_cpu_mamba_backend():
    """Make transformers' MambaMixer take its pure-PyTorch path instead of probing CUDA
    kernels. A no-op on a clean CPU-only box; needed only when the environment has a
    broken or version-mismatched causal_conv1d / mamba_ssm build (the "undefined symbol"
    ABI error). Mirrors research/vs_seg/infer_segmamba_cpu.py."""
    import sys
    import transformers  # noqa: F401
    for modname in ("transformers.utils.import_utils", "transformers.utils"):
        mod = sys.modules.get(modname)
        if mod is not None:
            for fn in ("is_causal_conv1d_available", "is_mamba_ssm_available"):
                if hasattr(mod, fn):
                    setattr(mod, fn, (lambda *a, **k: False))


RUN_SEGMAMBA_CPU = False
SEGMAMBA_WEIGHTS = "models/best_segmamba.pth"

if RUN_SEGMAMBA_CPU:
    _force_cpu_mamba_backend()  # skips the mismatched CUDA kernel probe (defined just above)
    from models_segmamba.segmambav2 import SegMamba

    # Pure-PyTorch Mamba backend: runs on CPU, no mamba_ssm / causal_conv1d needed.
    seg_model = SegMamba(
        in_chans=1, out_chans=2, depths=[2, 2, 2, 2],
        feat_size=[48, 96, 192, 384], hidden_size=768,
        mamba_backend="mambamixer",
    )
    # A mamba_ssm-trained checkpoint loads strict into the mambamixer model unchanged.
    from torch.nn.modules.utils import consume_prefix_in_state_dict_if_present
    sd = torch.load(SEGMAMBA_WEIGHTS, map_location="cpu")
    sd = sd["model"] if isinstance(sd, dict) and "model" in sd else sd
    consume_prefix_in_state_dict_if_present(sd, "_orig_mod.")  # official torch helper
    seg_model.load_state_dict(sd, strict=True)
    seg_model.eval()  # stays on CPU

    cpu_preds = patch_inference(
        learner=seg_model,
        config=patch_config,
        file_paths=NEW_CASES,
        pre_inference_tfms=pre_inference_tfms,
        save_dir=OUTPUT_DIR,
        progress=True,
        tta=False,   # TTA is impractical on CPU
    )
    print(f"SegMamba CPU: {len(cpu_preds)} prediction(s) written to {OUTPUT_DIR}/")
else:
    print("Set RUN_SEGMAMBA_CPU = True (with a SegMamba checkpoint) to run this path.")

## Recap and pointers

- **This notebook runs a 5-fold soft-voting ensemble.** Each fold's probability map is produced in the input's original space, the five are averaged, and only then decoded (argmax + keep-largest-component). This usually beats any single fold.
- **Preprocessing parity is the rule that governs correctness.** The prediction is valid only because inference reused the training `apply_reorder`, `target_spacing`, and foreground `ZNormalization`. When in doubt, load an exported learner (which carries its own preprocessing) or ship a `PatchConfig` JSON via `store_patch_variables` / `load_patch_variables`.
- **Patch inference** slides an overlapping grid, predicts per patch, and blends with a Hann window, so arbitrarily large volumes fit in memory.
- **TTA** trades roughly 8x compute for tighter borders; **AMP** speeds up the GPU forward pass. Neither changes the required preprocessing.
- **Model choice is orthogonal to this notebook.** UNet and DynUNet run anywhere; SegMamba needs the fork and prefers a GPU, with a documented CPU fallback.

Pointers:

- Training and the checkpoints this notebook consumes: `01_five_fold_cross_validation.ipynb`.
- An earlier whole-volume ensemble reference: `research/vs_seg/10_ensemble_inference_verification_unet.ipynb`.
- SegMamba CPU details: `research/vs_seg/infer_segmamba_cpu.py` and `research/vs_seg/SEGMAMBA_CPU.md`.
- The inference engine and config: `fastMONAI/vision_patch.py` (`patch_inference`, `PatchInferenceEngine`, `PatchConfig`).